# A tiny end-to-end model

This is the final link of the pipeline, and it is deliberately throwaway. The aim is not accuracy. It is to prove that prepared data can flow into a model and predictions come back out in the right shape. A working but useless model here means the whole chain, from audio to predictions, holds together. Real models and honest evaluation come later.

## Turning the data into tensors

A tensor is PyTorch's version of a numpy array, the format the model works in. A convolutional network expects its input shaped as (batch, channels, height, width). The spectrograms are (120, 128, 313), so a channels dimension is added to make them (120, 1, 128, 313). The single channel reflects that a spectrogram is greyscale, unlike a colour image which has three.

In [ ]:
# Import necessary libraries
import numpy as np
import json
import torch
import torch.nn as nn

# Load the toy dataset
specs = np.load('../data/toy_specs.npy')
Y = np.load('../data/toy_Y.npy')
with open('../data/toy_classes.json') as f:
    classes = json.load(f)

print("spectrograms:", specs.shape)
print("labels:", Y.shape)
print("species:", len(classes))

spectrograms: (120, 128, 313)
labels: (120, 21)
species: 21


## Turning the data into tensors

A tensor is PyTorch's version of a numpy array, the format the model works in. A convolutional network expects its input shaped as (batch, channels, height, width). The spectrograms are (120, 128, 313), so a channels dimension is added to make them (120, 1, 128, 313). The single channel reflects that a spectrogram is greyscale, unlike a colour image which has three.

In [7]:
# a CNN expects (batch, channels, height, width).
# our spectrograms are (120, 128, 313), so add a channel dimension -> (120, 1, 128, 313)
X = torch.tensor(specs, dtype=torch.float32).unsqueeze(1)
Y_t = torch.tensor(Y, dtype=torch.float32)

print("X shape:", X.shape)
print("Y shape:", Y_t.shape)

X shape: torch.Size([120, 1, 128, 313])
Y shape: torch.Size([120, 21])


## Defining the model

A deliberately minimal convolutional network. The convolution slides small filters over the spectrogram looking for patterns, the activation adds non-linearity so it can learn more than straight lines, the pooling squashes each filter down to a single value, and the final linear layer maps those to one score per species. It is the smallest thing that still counts as a CNN.

In [ ]:
class TinyCNN(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),  # find simple patterns
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),                    # squash to one number per filter
            nn.Flatten(),
            nn.Linear(8, n_classes),                    # map to one score per species
        )

    def forward(self, x):  # x is (batch, channels, height, width)
        return self.net(x)

# Instantiate the model and print its architecture
model = TinyCNN(len(classes))
print(model)

TinyCNN(
  (net): Sequential(
    (0): Conv2d(1, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): AdaptiveAvgPool2d(output_size=1)
    (3): Flatten(start_dim=1, end_dim=-1)
    (4): Linear(in_features=8, out_features=21, bias=True)
  )
)


## Training

The training loop is the core of it. Each pass over the data does four things: predict, measure how wrong the predictions are, work out which direction to adjust the weights, and apply that adjustment. `BCEWithLogitsLoss` is the correct loss for a multi-label task, where several species can be correct in the same segment. If the model is learning, the loss should fall across the epochs.

In [9]:
loss_fn = nn.BCEWithLogitsLoss()               # right loss for multi-label
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(10):
    optimizer.zero_grad()      # clear last step's gradients
    out = model(X)             # predictions for all 120 segments
    loss = loss_fn(out, Y_t)   # how wrong they are
    loss.backward()            # work out which way to nudge the weights
    optimizer.step()           # nudge them
    print(f"epoch {epoch+1}  loss {loss.item():.4f}")

epoch 1  loss 3.8029
epoch 2  loss 2.9987
epoch 3  loss 2.3530
epoch 4  loss 1.8173
epoch 5  loss 1.4790
epoch 6  loss 1.2376
epoch 7  loss 1.0025
epoch 8  loss 0.8051
epoch 9  loss 0.6825
epoch 10  loss 0.6500


## Predictions come out

The trained model is run on one segment, and the raw scores are passed through a sigmoid to turn them into probabilities between 0 and 1, one per species. With a toy model trained and tested on the same 120 examples, these numbers carry no real meaning. Their only job is to confirm predictions emerge in the right shape.

In [ ]:
model.eval()
with torch.no_grad():
    probs = torch.sigmoid(model(X[:1]))   # sigmoid turns scores into 0-1 probabilities

# Print the predicted probabilities for the first segment
print("prediction shape:", probs.shape)
print("first segment predicted probabilities:") 
print(probs[0].numpy().round(3))

prediction shape: torch.Size([1, 21])
first segment predicted probabilities:
[0.382 0.5   0.698 0.356 0.394 0.365 0.309 0.385 0.311 0.592 0.451 0.574
 0.75  0.657 0.16  0.565 0.5   0.321 0.341 0.245 0.462]


## What this shows

This is a deliberately tiny model trained on 120 segments from five files, then run on the same data. The predictions are not meaningful, and are not supposed to be. The purpose is only to prove the pipeline runs from end to end: audio is cut into segments, turned into spectrograms, paired with multi-hot labels, passed through a model, and predictions come back out in the right shape. The falling loss confirms the training loop works. Real models, a proper train and test split, and honest evaluation come later.